# Эксперименты по спарсификации модели

Ранее был обучени baseline и получены для него:
* Latency
* Throughput
* IoU

Будут проведены следующие  эксперименты:
1. baseline + unstructured pruning + fine-tune
2. baseline + structured pruning + fine-tune
3. baseline + 2:4 semi-structured pruning + fine-tune

После каждого эксперимента будут проведены замеры влияния на:
* метрику качества
* пропускную способность
* задержку

## Импорты и константы

In [ ]:
import zipfile
from pathlib import Path
from urllib.request import urlretrieve
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm
import torch
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
from pycocotools.coco import COCO
import numpy as np
from PIL import Image as PILImage
from torchvision.transforms import v2 as tr
from torch.utils.data import DataLoader, Dataset
from torchvision import tv_tensors

import colorsys
from torchvision.transforms import InterpolationMode
from torchvision.transforms.functional import pil_to_tensor

import pandas as pd
import torch.nn as nn
import copy
import torch.nn.utils.prune as prune

from collections import defaultdict
from termcolor import colored

import os
import heapq
import time

from IPython.display import clear_output

In [ ]:
ROOT_DATASET = "/data/datasets/coco"
CKPT_PATH = "../../../weights/checkpoint_coco_fp16.pt"

CKPT_DIR = "./weights/"
CKPT_TOPK_DIR = CKPT_DIR + "topk"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IGNORE_INDEX = 255

NUM_CLASSES = 81
IMAGE_SIZE = (384, 384)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

NUM_WORKERS = 4
BATCH_SIZE = 16

USE_AMP = torch.cuda.is_available()

USE_CUDA_AMP = USE_AMP and str(DEVICE).startswith("cuda")
USE_BF16 = USE_CUDA_AMP and torch.cuda.is_bf16_supported()
USE_FP16 = USE_CUDA_AMP and (not USE_BF16)


## Загрузка baseline

In [ ]:
ROOT = Path(ROOT_DATASET)
ROOT.mkdir(parents=True, exist_ok=True)

urls = {
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
    "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

def download(url, dst):
    if dst.exists():
        print(f"[skip] {dst.name} already exists")
        return
    print(f"[download] {dst.name}")
    urlretrieve(url, dst)
    print(f"[ok] {dst.name}")

def extract(zip_path, out_dir):
    marker = out_dir / (zip_path.stem.replace(".zip", ""))
    print(f"[extract] {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    print(f"[ok] extracted {zip_path.name}")

In [ ]:
# Это нужно запустить один раз для скачивания датасета!!!
# for name, url in urls.items():
#     zip_path = ROOT / name
#     download(url, zip_path)
#     extract(zip_path, ROOT)

In [ ]:
print("Done. Check structure:")
for p in [
    ROOT / "train2017",
    ROOT / "val2017",
    ROOT / "annotations" / "instances_train2017.json",
    ROOT / "annotations" / "instances_val2017.json",
]:
    print(p, "->", p.exists())

In [ ]:
def build_coco_semantic_masks(
    root=ROOT_DATASET,
    split="train2017",
    out_dir_name="semantic_masks",
    ignore_index=255,
    skip_crowd=True,
):
    """
    Создает PNG-маски semantic segmentation для COCO split.
    Маска: 0=background, 1..80=классы COCO (плотная перенумерация), 255=ignore (crowd).
    """
    root = Path(root)
    ann_file = root / "annotations" / f"instances_{split}.json"
    images_dir = root / split
    masks_dir = root / out_dir_name / split
    masks_dir.mkdir(parents=True, exist_ok=True)

    coco = COCO(str(ann_file))
    image_ids = sorted(coco.getImgIds())

    # cat_id -> train_id (1..N), 0 оставляем background
    cat_ids = sorted(coco.getCatIds())
    cat_id_to_train_id = {cat_id: i + 1 for i, cat_id in enumerate(cat_ids)}

    # сохранить mapping для повторного использования
    mapping_path = root / out_dir_name / "cat_id_to_train_id.npy"
    np.save(mapping_path, cat_id_to_train_id, allow_pickle=True)

    for image_id in tqdm(image_ids, desc=f"Building masks {split}"):
        img_info = coco.loadImgs(image_id)[0]
        h, w = img_info["height"], img_info["width"]
        out_path = masks_dir / (Path(img_info["file_name"]).stem + ".png")

        # если маска уже есть, можно пропустить
        if out_path.exists():
            continue

        mask = np.zeros((h, w), dtype=np.uint8)

        ann_ids = coco.getAnnIds(imgIds=image_id, iscrowd=None)
        anns = coco.loadAnns(ann_ids)
        anns = sorted(anns, key=lambda a: a.get("area", 0), reverse=True)

        for ann in anns:
            m = coco.annToMask(ann).astype(bool)

            if ann.get("iscrowd", 0) == 1:
                if skip_crowd:
                    mask[m] = ignore_index
                continue

            train_id = cat_id_to_train_id[ann["category_id"]]  # 1..80
            mask[m] = train_id

        PILImage.fromarray(mask, mode="L").save(out_path)

    print(f"[OK] {split}: masks in {masks_dir}")

In [ ]:
# Запуск (один раз):
# build_coco_semantic_masks(root=ROOT_DATASET, split="train2017")
# build_coco_semantic_masks(root=ROOT_DATASET, split="val2017")

In [ ]:
def load_baseline():
    model = smp.Unet(
        encoder_name="resnet101",
        encoder_weights=None,
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None,
    )
    # baseline = model.half() # fp16
    baseline = model.bfloat16() # bf16

    state = torch.load(CKPT_PATH, map_location="cpu")
    baseline.load_state_dict(state, strict=True)

    baseline = baseline.to(DEVICE).eval()

    return baseline


In [ ]:
baseline = load_baseline()

## Формирование датасета

In [ ]:
class SegTransform:
    '''
    Класс-преобразователь, который применяет аугментации к паре (image, mask)
    '''
    
    def __init__(self, size, train=True):

        self.mean = IMAGENET_MEAN
        self.std = IMAGENET_STD

        if train:
            self.joint_prepare = tr.Compose([
                tr.RandomResizedCrop(
                    size,
                    scale=(0.6, 1.0),
                    interpolation=InterpolationMode.BILINEAR,
                    antialias=True
                ),
                tr.RandomHorizontalFlip(p=0.5),
            ])
        else:
            self.joint_prepare = tr.Compose([
                tr.Resize(size, interpolation=InterpolationMode.BILINEAR, antialias=True),
            ])

        if train:
            self.image_prepare = tr.Compose([
                tr.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
                tr.ToDtype(torch.float32, scale=True),
                tr.Normalize(mean=self.mean, std=self.std),
            ])
        else:
            self.image_prepare = tr.Compose([
                tr.ToDtype(torch.float32, scale=True),
                tr.Normalize(mean=self.mean, std=self.std),
            ])

    def __call__(self, image, mask):
        image = tv_tensors.Image(image)
        mask = tv_tensors.Mask(pil_to_tensor(mask).squeeze(0).to(torch.int64))  # без np.array

        image, mask = self.joint_prepare(image, mask)
        image = self.image_prepare(image)
        return image, mask.long()

In [ ]:
class COCOSemanticDataset(Dataset):
    """
    Semantic-seg dataset на базе COCO instances.
    Возвращает:
      image: PIL.Image (RGB)
      mask:  PIL.Image (L), где:
             0 = background
             1..N = категории COCO (плотная перенумерация)
             ignore_index (по умолчанию 255) можно использовать при необходимости
    """

    def __init__(
        self,
        root=ROOT_DATASET,
        split="train2017",
        masks_root="semantic_masks",
        ann_file=None,
        transforms=None,
        ignore_index=IGNORE_INDEX,
    ):
        self.root = Path(root)
        self.split = split
        self.transforms = transforms
        self.ignore_index = ignore_index

        self.images_dir = self.root / split
        self.masks_dir = self.root / masks_root / split

        if ann_file is None:
            ann_file = self.root / "annotations" / f"instances_{split}.json"
        self.coco = COCO(str(ann_file))

        self.image_ids = sorted(self.coco.getImgIds())

        cat_ids = sorted(self.coco.getCatIds())
        cats = self.coco.loadCats(cat_ids)
        self.class_names = ["background"] + [c["name"] for c in sorted(cats, key=lambda x: x["id"])]
        self.num_classes = len(self.class_names)

        # Проверка наличия масок
        missing = 0
        for image_id in self.image_ids[:500]:  # быстрая проверка части
            info = self.coco.loadImgs(image_id)[0]
            mpath = self.masks_dir / (Path(info["file_name"]).stem + ".png")
            if not mpath.exists():
                missing += 1
        if missing > 0:
            raise FileNotFoundError(
                f"Missing precomputed masks in {self.masks_dir}. "
                f"Run build_coco_semantic_masks(...) first."
            )

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.coco.loadImgs(image_id)[0]

        image_path = self.images_dir / info["file_name"]
        mask_path = self.masks_dir / (Path(info["file_name"]).stem + ".png")

        with PILImage.open(image_path) as im:
            image = im.convert("RGB").copy()

        with PILImage.open(mask_path) as mm:
            mask = mm.convert("L").copy()

        if self.transforms is not None:
            image, mask = self.transforms(image, mask)

        return image, mask

In [ ]:
train_ds = COCOSemanticDataset(
    root=ROOT_DATASET,
    split="train2017",
    masks_root="semantic_masks",
    transforms=SegTransform(size=IMAGE_SIZE, train=True),
)

val_ds = COCOSemanticDataset(
    root=ROOT_DATASET,
    split="val2017",
    masks_root="semantic_masks",
    transforms=SegTransform(size=IMAGE_SIZE, train=False),
)

In [ ]:
def seg_collate_fn(batch):
    images, masks = zip(*batch)
    images = torch.stack(images, dim=0)
    masks = torch.stack(masks, dim=0)
    return images, masks

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=2,
    collate_fn=seg_collate_fn,
    # worker_init_fn=seed_worker,
    # generator=seed_generator,
    timeout= 60,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=2,
    collate_fn=seg_collate_fn,
    # worker_init_fn=seed_worker,
    # generator=seed_generator,
    timeout= 60,
)

## Получение примеров предсказаний


In [ ]:
def _build_seg_cmap_with_ignore(num_classes: int, ignore_color=(0.15, 0.15, 0.15, 1.0)):
    hues = (np.arange(num_classes) * 0.61803398875) % 1.0
    colors = []
    for h in hues:
        r, g, b = colorsys.hsv_to_rgb(h, 0.75, 1.0)
        colors.append((r, g, b, 1.0))

    colors.append(ignore_color)
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(np.arange(-0.5, num_classes + 1.5, 1), cmap.N)
    return cmap, norm

def _mask_boundaries(m: np.ndarray) -> np.ndarray:
    b = np.zeros_like(m, dtype=bool)
    b[:-1, :] |= (m[:-1, :] != m[1:, :])
    b[:, :-1] |= (m[:, :-1] != m[:, 1:])
    return b

def _boundary_rgba(mask: np.ndarray, color=(1.0, 1.0, 1.0, 1.0)):
    bd = _mask_boundaries(mask)
    overlay = np.zeros((*bd.shape, 4), dtype=np.float32)
    overlay[bd] = color
    return overlay

In [ ]:
def show_val_predictions_triplets(
    model,
    val_ds,
    device,
    n_samples: int = 4,
    seed: int = 42,
    ignore_index: int = IGNORE_INDEX,
    max_legend_classes: int = 10,
):
    model.eval()
    rng = random.Random(seed)
    idxs = [rng.randrange(len(val_ds)) for _ in range(n_samples)]

    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    num_classes = val_ds.num_classes
    ignore_show_id = num_classes
    cmap, norm = _build_seg_cmap_with_ignore(num_classes)

    fig, axes = plt.subplots(
        n_samples, 4, figsize=(16, 4 * n_samples),
        gridspec_kw={"width_ratios": [1, 1, 1, 0.9]}
    )
    if n_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    param_dtype = next(model.parameters()).dtype

    with torch.no_grad():
        for i, idx in enumerate(idxs):
            image, true_mask = val_ds[idx]
            x = image.unsqueeze(0).to(device, non_blocking=True)

            if param_dtype in (torch.float16, torch.bfloat16):
                x = x.to(dtype=param_dtype)

            logits = model(x)
            pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()

            img_show = (image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
            true_show = true_mask.cpu().numpy().copy()
            true_show[true_show == ignore_index] = ignore_show_id

            axes[i, 0].imshow(img_show)
            axes[i, 0].set_title(f"image #{idx}")
            axes[i, 0].axis("off")

            axes[i, 1].imshow(true_show, cmap=cmap, norm=norm, interpolation="nearest")
            axes[i, 1].imshow(_boundary_rgba(true_show), interpolation="nearest")
            axes[i, 1].set_title("true_mask_val")
            axes[i, 1].axis("off")

            axes[i, 2].imshow(pred_mask, cmap=cmap, norm=norm, interpolation="nearest")
            axes[i, 2].imshow(_boundary_rgba(pred_mask), interpolation="nearest")
            axes[i, 2].set_title("pred_mask")
            axes[i, 2].axis("off")

            # Легенда только по классам этой тройки
            ids, counts = np.unique(np.concatenate([true_show.ravel(), pred_mask.ravel()]), return_counts=True)
            order = np.argsort(-counts)
            ids = ids[order]

            shown = ids[:max_legend_classes]
            handles = []
            for cid in shown:
                cid = int(cid)
                if cid == ignore_show_id:
                    label = "255: ignore"
                else:
                    label = f"{cid}: {val_ds.class_names[cid]}"
                handles.append(Patch(facecolor=cmap(cid), edgecolor="none", label=label))                

            axes[i, 3].axis("off")
            axes[i, 3].legend(handles=handles, loc="upper left", frameon=False, fontsize=8, handlelength=1.0)
            extra = len(ids) - len(shown)
            axes[i, 3].set_title(f"classes in sample{f' (+{extra} more)' if extra > 0 else ''}", fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
show_val_predictions_triplets(
    model=baseline,
    val_ds=val_ds,
    device=DEVICE,
    n_samples=4,
    seed=13,
    ignore_index=IGNORE_INDEX,
)

## Получение метрик 

In [ ]:
@torch.no_grad()
def per_class_iou(
    model,
    loader,
    device,
    class_names,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
):
    model.eval()
    confmat = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)

    first_param = next(model.parameters(), None)
    param_dtype = first_param.dtype if first_param is not None else torch.float32

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).long()

        if param_dtype in (torch.float16, torch.bfloat16):
            images = images.to(dtype=param_dtype)

        logits = model(images)
        preds = logits.argmax(dim=1).long()

        t = masks.reshape(-1)
        p = preds.reshape(-1)

        valid = t != ignore_index
        valid &= (t >= 0) & (t < num_classes)
        valid &= (p >= 0) & (p < num_classes)

        t = t[valid]
        p = p[valid]
        if t.numel() == 0:
            continue

        idx = t * num_classes + p
        confmat += torch.bincount(idx, minlength=num_classes**2).reshape(num_classes, num_classes)

    tp = confmat.diag()
    fp = confmat.sum(0) - tp
    fn = confmat.sum(1) - tp
    denom = tp + fp + fn

    iou = tp.float() / torch.clamp(denom.float(), min=1.0)
    present = denom > 0
    miou_present = iou[present].mean().item() if present.any() else 0.0

    rows = []
    for i in range(num_classes):
        rows.append({
            "class_id": i,
            "class_name": class_names[i],
            "iou": float(iou[i].item()),
            "present_in_val": bool(present[i].item()),
            "pixels_gt_or_pred": int(denom[i].item()),
        })

    return rows, miou_present


In [ ]:
def plot_compare_model(
    baseline,
    modified_model,
    loader,
    device,
    class_names,
    per_class_iou_fn,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
):
    rows_baseline, miou_baseline = per_class_iou_fn(
        baseline, loader, device, class_names, num_classes=num_classes, ignore_index=ignore_index
    )
    rows_modified, miou_modified = per_class_iou_fn(
        modified_model, loader, device, class_names, num_classes=num_classes, ignore_index=ignore_index
    )

    iou_baseline = np.array([r["iou"] for r in rows_baseline], dtype=np.float32)
    iou_modified = np.array([r["iou"] for r in rows_modified], dtype=np.float32)
    delta_percent = (miou_modified - miou_baseline) * 100

    idx = np.arange(num_classes)

    width = 0.42
    x = np.arange(len(idx))

    plt.figure(figsize=(max(12, len(idx) * 0.35), 6))
    plt.bar(x - width / 2, iou_baseline[idx], width=width, label="baseline")
    plt.bar(x + width / 2, iou_modified[idx], width=width, label="modified_model")
    plt.xticks(x, [class_names[i] for i in idx], rotation=90)
    plt.ylabel("IoU")
    plt.title(
        f"Per-class IoU: baseline vs modified_model\nmIoU baseline={miou_baseline:.4f}, pruning={miou_modified:.4f}, delta={delta_percent:+.4f}"
    )
    plt.grid(axis="y", alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_compare_model(
    baseline=baseline,
    modified_model=baseline, # Для нормальной работы функции
    loader=val_loader,
    device=DEVICE,
    class_names=val_ds.class_names,
    per_class_iou_fn=per_class_iou,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
)

## Вспомогательные функции для экспериментов

In [ ]:
def beautiful_int(i):
    i = str(i)
    return ".".join(reversed([i[max(j, 0):j+3] for j in range(len(i) - 3, -3, -3)]))

# Считаем общее число параметров в нашей модели
def model_num_params(model, verbose_all=True, verbose_only_learnable=False):
    sum_params = 0
    sum_learnable_params = 0
    submodules = defaultdict(lambda : [0, 0])
    for name, param in model.named_parameters():
        num_params = np.prod(param.shape)
        if verbose_all or (verbose_only_learnable and param[1].requires_grad):
            print(
                colored(
                    '{: <42} ~  {: <9} params ~ grad: {}'.format(
                        name,
                        beautiful_int(num_params),
                        param.requires_grad,
                    ),
                    {True: "green", False: "red"}[param[1].requires_grad],
                )
            )
        sum_params += num_params
        sm = name.split(".")[0]
        submodules[sm][0] += num_params
        if param.requires_grad:
            sum_learnable_params += num_params
            submodules[sm][1] += num_params
    print(
        f'\nIn total:\n  - {beautiful_int(sum_params)} params\n  - {beautiful_int(sum_learnable_params)} learnable params'
    )
    
    for sm, v in submodules.items():
        print(
            f"\n . {sm}:\n .   - {beautiful_int(submodules[sm][0])} params\n .   - {beautiful_int(submodules[sm][1])} learnable params"
        )
    return sum_params, sum_learnable_params


## Unstructured pruning

Метод подразумевает:
* Зануление отдельных весов внутри тензоров весов (обычно самые маленькие по модулю)
* Форма слоя не меняется

Для метода выберем процент для прунинга:
* 10%
* 20%
* 30%
* 40%

Применим Unstructured pruning для:
* только декодера
* всей модели

После применения метода проверим насколько pruning ухудшил модель до fine-tuning:
* посчитаем val_loss
* посчитаем mIoU
* сравним с baseline

Применим fine-tuning для нивелирования ущерба:
* несколько эпох
* уменьшим learning rate в 5-10 раз относительно baseline
* сохраним лучшую модель по val_mIoU без маски

### Анализ структуры модели и получение подходящих слоёв для эксперимента

In [ ]:
sum_params, sum_learnable_params = model_num_params(baseline)

Видим, что в модели 51.524.833 обучаемых параметра. Из них:
* encoder - 42.500.160
* decoder - 9.012.928
* segmentation_head - 11.745

Для unstructured pruning обычно кандидаты - это:
* веса сверточных слоёв
* веса линейных слоёв
* слои в которых сосредоточена основная масса параметров
* слои, которые не слишком чувствительным к pruning

Для поиска подходящих слоёв проведем эксперимент:
* возьмем самые большие сверточные и линейные слои
* применим pruning только к конкретному слою по отдельности, а также с разными amount
* проверим деградацию с baseline по mIoU
* сохраним изменение в список (имя слоя, число параметров, sparsity, amount, delta mIoU)
* выберем вручную подходящие слои для совместного pruning по сочетанию размера слоя и устойчивости к pruning

#### Выберем все большие слои для эксперимента

In [ ]:
import torch.nn as nn
import pandas as pd

def get_largest_layers_info(model, part, min_size, layer_type):
    if part not in {"encoder", "decoder"}:
        raise ValueError("part must be 'encoder' or 'decoder'")
    if layer_type not in {"conv", "linear"}:
        raise ValueError("layer_type must be 'conv' or 'linear'")

    if layer_type == "conv":
        types = (nn.Conv2d,)
    else:
        types = (nn.Linear,)

    rows = []
    for name, module in model.named_modules():
        if name == "":
            continue
        if part not in name.split("."):
            continue
        if not isinstance(module, types):
            continue

        n_params = sum(p.numel() for p in module.parameters(recurse=False))
        if n_params >= min_size:
            rows.append({
                "part": part,
                "layer_type": layer_type,
                "layer_name": name,
                "n_params": n_params,
            })
    return rows


In [ ]:
min_size_layer=500_000

result = []
for part in ["encoder", "decoder"]:
    for layer_type in ["conv", "linear"]:
        result += get_largest_layers_info(baseline, part, min_size_layer, layer_type)

result_df = pd.DataFrame(result).sort_values(
    ["part", "layer_type", "n_params"], ascending=[True, True, False]
).reset_index(drop=True)

ratio = result_df["n_params"].sum() / sum_params * 100
print(f"Доля параметров выбранных слоев от модели в целом: {ratio:.4}%")
result_df

In [ ]:
def apply_unstructured_pruning(model, name_layer, amount):
    model_copy = copy.deepcopy(model)

    layer = model_copy.get_submodule(name_layer)

    prune.l1_unstructured(layer, name="weight", amount=amount)

    return model_copy


In [ ]:
_, mIoU_baseline = per_class_iou(baseline, val_loader, DEVICE, val_ds.class_names)
print(f"mIoU_baseline: {mIoU_baseline*100:.6}%")

#### Расчёт ущерба для единичного слоя

In [ ]:
cache_path = Path("experiments_df.csv")

if cache_path.exists():
    experiments_df = pd.read_csv(cache_path)
    print(f"Загружено из {cache_path}")
else:
    # amounts = [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]
    amounts = [0.1, 0.2, 0.3, 0.4, 0.5]
    rows = []
    for _, row in result_df[["layer_name", "n_params"]].drop_duplicates().iterrows():
        layer_name = row["layer_name"]
        n_params = int(row["n_params"])
        
        layer_result = {
            "layer_name": layer_name,
            "n_params": n_params,
        }

        for amount in amounts:
            pruned_model = apply_unstructured_pruning(baseline, layer_name, amount=amount)
            _, mIoU_pruned = per_class_iou(pruned_model, val_loader, DEVICE, val_ds.class_names)
            delta_percent = (mIoU_pruned - mIoU_baseline) * 100
            layer_result[f"amount={amount}"] = delta_percent
        
        rows.append(layer_result)
    
    experiments_df = pd.DataFrame(rows)
    experiments_df.to_csv(cache_path, index=False)
    print(f"Сохранено в {cache_path}")

experiments_df


In [ ]:
def build_layer_pruning_plan(
    experiments_df: pd.DataFrame,
    params_step: int = 500_000,
    miou_percent_for_params_step: float = 0.001
) -> dict[str, float]:

    amount_cols = sorted(
        [c for c in experiments_df.columns if c.startswith("amount=")],
        key=lambda c: float(c.split("=")[1]),
    )

    df = experiments_df.copy()
    df = df.sort_values("n_params", ascending=False)

    plan = {}
    for _, row in df.iterrows():
        n_params = float(row["n_params"])
        layer = row["layer_name"]

        allowed_damage = -n_params / params_step * miou_percent_for_params_step
        # print(layer, n_params, allowed_damage)

        best_amount = None
        for col in amount_cols:
            amount = float(col.split("=")[1])
            miou_damage = row[col]
            if miou_damage > allowed_damage:
                best_amount = amount
            # print(amount, miou_damage)

        if best_amount is not None:
            plan[layer] = best_amount

    return plan


In [ ]:
# Выбрать руками
# layer_name_unstructured_pruning = {
#     "decoder.blocks.0.conv1.0":         0.3,
#     "decoder.blocks.1.conv1.0":         0.2,
#     "encoder.layer4.0.conv2":           0.3,
#     "encoder.layer4.1.conv2":           0.4,
#     "encoder.layer4.2.conv2":           0.5,
#     "encoder.layer4.0.downsample.0":    0.4,
#     "encoder.layer4.0.conv3":           0.5,
#     "encoder.layer4.1.conv1":           0.3,
#     "encoder.layer4.1.conv3":           0.5,
#     "encoder.layer4.2.conv1":           0.3,
#     "encoder.layer4.2.conv3":           0.2
#     }

# Или через функцию с порогом в 0,01 процента на каждые +- 500 тысяч параметров
layer_name_unstructured_pruning = build_layer_pruning_plan(experiments_df, 500_000, 0.01)
layer_name_unstructured_pruning

### Эксперимент

In [ ]:
def apply_unstructured_pruning_plan(model, layer_name_unstructured_pruning):
    model_copy = copy.deepcopy(model)

    items = layer_name_unstructured_pruning.items()

    for layer_name, amount in items:
        layer = model_copy.get_submodule(layer_name)

        if not hasattr(layer, "weight"):
            raise ValueError(f"Layer '{layer_name}' has no attribute 'weight'.")

        if not isinstance(layer, (nn.Linear, nn.Conv1d, nn.Conv2d, nn.Conv3d)):
            raise ValueError(
                f"Layer '{layer_name}' is {type(layer).__name__}, expected Conv/Linear."
            )

        prune.l1_unstructured(layer, name="weight", amount=amount)
        # Если нужно убрать reparameterization и оставить обычный weight:
        # prune.remove(layer, "weight")

    return model_copy


In [ ]:
def zero_fraction(model):
    zero = 0
    total = 0

    target_types = (nn.Linear, nn.Conv2d,)
    modules = [m for m in model.modules() if isinstance(m, target_types) and hasattr(m, "weight")]
    tensors = [m.weight for m in modules]

    with torch.no_grad():
        for w in tensors:
            total += w.numel()
            zero += (w == 0).sum().item()

    return zero / total if total > 0 else 0.0


In [ ]:
pruned_model = apply_unstructured_pruning_plan(baseline, layer_name_unstructured_pruning)

In [ ]:
s_global = zero_fraction(pruned_model)
print(f"Обнулено {s_global:.2%} весов сверточных и линейных слоёв (но их не оказалось)")


In [ ]:
plot_compare_model(
    baseline=baseline,
    modified_model=pruned_model,
    loader=val_loader,
    device=DEVICE,
    class_names=val_ds.class_names,
    per_class_iou_fn=per_class_iou
)

In [ ]:
def make_pruning_permanent(model):
    for module in model.modules():
        if hasattr(module, "weight_mask"):
            prune.remove(module, "weight")
            # print(module)
    return model

In [ ]:
pruned_model = make_pruning_permanent(pruned_model)

save_path = Path("../../../weights/pruned_unstructed.pt")
torch.save(pruned_model.state_dict(), save_path)

### Вывод

Модель сохраняет устойчивость и качество при занулении большой доли весов в больших сверточных слоях, что говорит о наличии параметрической избыточности (что и было целью этого эксперимента). Но такой тип  спарсификации не меняет архитектуру и в стандартном runtime не даёт заметного ускорения инференса.

Этот результат был ожидаем.

## Structured pruning

В предыдущем эксперименте мы доказали параметрическую избыточность модели.

Самые большие слои в оптимизируемой модели - это свёрточные слои. Поэтому в эксперименте будем удалять часть входных или выходных каналов фильтров.

Такой подход уже меняет архитектуру и может дать реальное ускорение.

In [ ]:
import copy
import torch.nn as nn
import torch.nn.utils.prune as prune

def apply_structured_pruning(model, layer_name, amount, n=1, dim=0):
    """
    n=1 -> L1-norm, n=2 -> L2-norm
    dim=0 -> prune output channels (обычно то, что нужно для Conv2d)
    """
    m = copy.deepcopy(model)
    layer = m.get_submodule(layer_name)

    if not isinstance(layer, nn.Conv2d):
        raise TypeError(f"{layer_name} is {type(layer)}, expected nn.Conv2d")

    prune.ln_structured(layer, name="weight", amount=amount, n=n, dim=dim)
    return m

def apply_structured_pruning_plan(model, pruning_plan, n=1, dim=0):
    """
    pruning_plan: dict[layer_name -> amount]
    """
    m = copy.deepcopy(model)
    for layer_name, amount in pruning_plan.items():
        layer = m.get_submodule(layer_name)
        if isinstance(layer, nn.Conv2d):
            prune.ln_structured(layer, name="weight", amount=amount, n=n, dim=dim)
    return m

def make_pruning_permanent(model):
    for module in model.modules():
        if hasattr(module, "weight_mask"):
            prune.remove(module, "weight")
    return model


In [ ]:
m_struct_prune = apply_structured_pruning(baseline, "encoder.layer4.2.conv2", 0.2)

In [ ]:
s_global = zero_fraction(m_struct_prune)
print(f"Обнулено {s_global:.2%} весов сверточных и линейных слоёв (но их не оказалось)")


In [ ]:
plot_compare_model(
    baseline=baseline,
    modified_model=m_struct_prune,
    loader=val_loader,
    device=DEVICE,
    class_names=val_ds.class_names,
    per_class_iou_fn=per_class_iou
)

### torch-tuning

In [ ]:
import copy
import torch
import torch.nn as nn
import torch_pruning as tp

# layer_name_structured_pruning = {
#     "decoder.blocks.0.conv1.0":         0.3,
#     "decoder.blocks.1.conv1.0":         0.2,
#     "encoder.layer4.0.conv2":           0.3,
#     "encoder.layer4.1.conv2":           0.4,
#     "encoder.layer4.2.conv2":           0.5,
#     "encoder.layer4.0.downsample.0":    0.4,
#     "encoder.layer4.0.conv3":           0.5,
#     "encoder.layer4.1.conv1":           0.3,
#     "encoder.layer4.1.conv3":           0.5,
#     "encoder.layer4.2.conv1":           0.3,
#     "encoder.layer4.2.conv3":           0.2,
# }

layer_name_structured_pruning = build_layer_pruning_plan(experiments_df, 500_000, 0.001)

def get_example_inputs(val_loader, device):
    batch = next(iter(val_loader))
    if isinstance(batch, (list, tuple)):
        x = batch[0]
    elif isinstance(batch, dict):
        x = batch.get("image", next(iter(batch.values())))
    else:
        x = batch
    return x[:1].to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def build_ch_sparsity_dict(model, plan):
    ch_sparsity_dict = {}
    missing = []
    bad_type = []
    for layer_name, amount in plan.items():
        try:
            module = model.get_submodule(layer_name)
        except Exception:
            missing.append(layer_name)
            continue
        if not isinstance(module, nn.Conv2d):
            bad_type.append((layer_name, type(module).__name__))
            continue
        ch_sparsity_dict[module] = float(amount)
    return ch_sparsity_dict, missing, bad_type

def auto_ignored_layers(model, num_classes=None):
    ignored = []
    for name, m in model.named_modules():
        low = name.lower()
        if any(k in low for k in ["segmentation_head", "classifier", "aux_head", "logits", "final"]):
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                ignored.append(m)
        if num_classes is not None and isinstance(m, nn.Conv2d):
            if m.out_channels == num_classes and m.kernel_size == (1, 1):
                ignored.append(m)
    return list(dict.fromkeys(ignored))


In [ ]:
layer_name_structured_pruning = build_layer_pruning_plan(experiments_df, 500_000, 0.1)
layer_name_structured_pruning

In [ ]:
m_tp = copy.deepcopy(baseline).eval().to(DEVICE)  # модель bf16
model_dtype = next(m_tp.parameters()).dtype
example_inputs = get_example_inputs(val_loader, DEVICE).to(
    DEVICE, dtype=model_dtype
)

# 3) Словарь layer->amount в формат module->amount
ch_sparsity_dict, missing, bad_type = build_ch_sparsity_dict(m_tp, layer_name_structured_pruning)
print("missing layers:", missing)
print("non-conv layers:", bad_type)

# 4) Какие слои не трогаем (голову сегментации)
ignored_layers = auto_ignored_layers(m_tp, num_classes=len(val_ds.class_names))

# 5) Pruner
importance = tp.importance.MagnitudeImportance(p=2)
pruner = tp.pruner.MagnitudePruner(
    m_tp,
    example_inputs=example_inputs,
    importance=importance,
    iterative_steps=6,      # один шаг = применить ratios из словаря
    pruning_ratio=0.0,        # глобально не пруним
    pruning_ratio_dict=ch_sparsity_dict,  # только указанные слои
    ignored_layers=ignored_layers,
    round_to=8,             # удобно для latency на GPU/CPU
)

# До pruning
base_macs, base_params = tp.utils.count_ops_and_params(copy.deepcopy(baseline).eval().to(DEVICE), example_inputs)

# 6) Физическое удаление каналов (изменяет архитектуру)
pruner.step()

# После pruning
pruned_macs, pruned_params = tp.utils.count_ops_and_params(m_tp, example_inputs)
print(f"Params: {base_params:,} -> {pruned_params:,} ({(1 - pruned_params/base_params)*100:.2f}% reduced)")
print(f"MACs  : {base_macs:,} -> {pruned_macs:,} ({(1 - pruned_macs/base_macs)*100:.2f}% reduced)")


In [ ]:
plot_compare_model(
    baseline=baseline,
    modified_model=m_tp,
    loader=val_loader,
    device=DEVICE,
    class_names=val_ds.class_names,
    per_class_iou_fn=per_class_iou
)

In [ ]:
model_num_params(m_tp)

### Дообучение


In [ ]:
@torch.no_grad()
def _update_confmat(confmat, logits, target, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    # logits: [B, C, H, W], target: [B, H, W]
    pred = logits.argmax(dim=1)
    valid = target != ignore_index
    pred = pred[valid]
    target = target[valid]
    if target.numel() == 0:
        return confmat

    idx = target * num_classes + pred
    binc = torch.bincount(idx, minlength=num_classes * num_classes)
    confmat += binc.reshape(num_classes, num_classes)
    return confmat


@torch.no_grad()
def _miou_from_confmat(confmat):
    # IoU_c = TP / (TP + FP + FN)
    tp = confmat.diag()
    fp = confmat.sum(dim=0) - tp
    fn = confmat.sum(dim=1) - tp
    denom = tp + fp + fn
    iou = tp.float() / torch.clamp(denom.float(), min=1.0)
    return iou.mean().item(), iou.cpu()


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    max_bad_batches=500,
    grad_clip=1.0,
    epoch_idx=None,
):
    model.train()
    processed_samples = 0
    bad_batches = 0

    confmat = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

    use_cuda_amp = USE_AMP and str(device).startswith("cuda")
    amp_dtype = (
        torch.bfloat16
        if use_cuda_amp and torch.cuda.is_bf16_supported()
        else torch.float16
    )
    # GradScaler используем только для fp16
    use_scaler = scaler is not None and use_cuda_amp and amp_dtype == torch.float16

    running_loss_sum = torch.zeros((), device=device)

    for bidx, (images, masks) in enumerate(tqdm(loader, mininterval=2.0)):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        if masks.dtype != torch.long:
            masks = masks.long()
            print("bidx=", bidx, " masks.dtype != torch.long:")

        valid = masks != ignore_index
        if not valid.any():
            continue

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_cuda_amp):
            logits = model(images)
            loss = criterion(logits, masks)

        if not torch.isfinite(loss):
            bad_batches += 1
            print(f"[bad-loss] epoch={epoch_idx} batch={bidx} loss={loss.detach().item()}")
            if bad_batches >= max_bad_batches:
                raise RuntimeError("Too many bad batches: non-finite loss")
            continue

        if use_scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
        else:
            loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=grad_clip,
            error_if_nonfinite=False,
        )
        if not torch.isfinite(grad_norm):
            bad_batches += 1
            print(f"[bad-grad] epoch={epoch_idx} batch={bidx} grad_norm={float(grad_norm)}")
            optimizer.zero_grad(set_to_none=True)
            if bad_batches >= max_bad_batches:
                raise RuntimeError("Too many bad batches: non-finite gradients")
            continue

        if use_scaler:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        bs = images.size(0)
        running_loss_sum += loss.detach() * bs
        processed_samples += bs
        _update_confmat(confmat, logits, masks, num_classes, ignore_index)

    if processed_samples == 0:
        raise RuntimeError(f"All batches skipped in epoch={epoch_idx}")

    epoch_loss = (running_loss_sum / processed_samples).item()
    epoch_miou, _ = _miou_from_confmat(confmat)

    if bad_batches > 0:
        print(f"[warn] epoch={epoch_idx} bad_batches={bad_batches}")

    return epoch_loss, epoch_miou

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    model.eval()

    confmat = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)
    running_loss_sum = torch.zeros((), device=device)
    processed_samples = 0

    use_cuda_amp = USE_AMP and str(device).startswith("cuda")
    amp_dtype = (
        torch.bfloat16
        if use_cuda_amp and torch.cuda.is_bf16_supported()
        else torch.float16
    )

    for bidx, (images, masks) in enumerate(tqdm(loader, mininterval=2.0)):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).long()

        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_cuda_amp):
            logits = model(images)
            loss = criterion(logits, masks)

        valid = masks != ignore_index
        if not valid.any():
            continue

        t = masks[valid]
        t_min = int(t.min().item())
        t_max = int(t.max().item())
        if t_min < 0 or t_max >= num_classes:
            raise RuntimeError(f"[eval bad-target] batch={bidx} min={t_min} max={t_max}")

        if not torch.isfinite(loss):
            raise RuntimeError(f"[eval bad-loss] batch={bidx} loss={loss.detach().item()}")

        bs = images.size(0)
        running_loss_sum += loss.detach() * bs
        processed_samples += bs
        _update_confmat(confmat, logits, masks, num_classes, ignore_index)

    if processed_samples == 0:
        raise RuntimeError("All eval batches skipped")

    epoch_loss = (running_loss_sum / processed_samples).item()
    epoch_miou, class_iou = _miou_from_confmat(confmat)
    return epoch_loss, epoch_miou, class_iou


In [ ]:
def save_topk_checkpoint(
    model,
    epoch,
    train_miou,
    val_miou,
    history,
    optimizer=None,
    scheduler=None,
    scaler=None,
    ckpt_dir=CKPT_TOPK_DIR,
    top_k=3,
    heap_state=None,
):
    os.makedirs(ckpt_dir, exist_ok=True)
    if heap_state is None:
        heap_state = []  # (val_miou, path)

    ckpt_path = os.path.join(
        ckpt_dir, f"epoch_{epoch:03d}_val_{val_miou:.4f}_train_{train_miou:.4f}.pt"
    )

    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "train_miou": float(train_miou),
        "val_miou": float(val_miou),
        "history": history,
    }

    if len(heap_state) < top_k:
        torch.save(payload, ckpt_path)
        heapq.heappush(heap_state, (val_miou, ckpt_path))
    else:
        worst_miou, worst_path = heap_state[0]
        if val_miou > worst_miou:
            heapq.heapreplace(heap_state, (val_miou, ckpt_path))
            if os.path.exists(worst_path):
                os.remove(worst_path)
            torch.save(payload, ckpt_path)

    return heap_state

In [ ]:
def learning_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs,
    device,
    scaler=None,
    scheduler=None,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    verbose=True,
    plot_live=True,
):
    """Полный цикл обучения + валидация + live-графики + best checkpoint."""

    if scaler is None:
        scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16) if USE_FP16 else None

    history = {
        "train_loss": [],
        "train_miou": [],
        "val_loss": [],
        "val_miou": [],
        "lr": [],
    }
    
    best_val_miou = -1.0
    best_state = copy.deepcopy(model.state_dict())

    def _plot_history():
        ep = range(1, len(history["train_loss"]) + 1)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].plot(ep, history["train_loss"], label="train")
        axes[0].plot(ep, history["val_loss"], label="val")
        axes[0].set_title("Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        axes[1].plot(ep, history["train_miou"], label="train")
        axes[1].plot(ep, history["val_miou"], label="val")
        axes[1].set_title("mIoU")
        axes[1].set_xlabel("Epoch")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        axes[2].plot(ep, history["lr"], label="lr")
        axes[2].set_title("Learning Rate")
        axes[2].set_xlabel("Epoch")
        axes[2].grid(True, alpha=0.3)
        axes[2].legend()

        plt.tight_layout()
        plt.show()

    topk_heap = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        train_loss, train_miou = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            num_classes=num_classes,
            ignore_index=ignore_index,
            epoch_idx=epoch
        )

        val_loss, val_miou, _ = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            num_classes=num_classes,
            ignore_index=ignore_index,
        )

        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["train_miou"].append(train_miou)
        history["val_loss"].append(val_loss)
        history["val_miou"].append(val_miou)
        history["lr"].append(current_lr)

        if scheduler is not None:
            scheduler.step()

        if val_miou > best_val_miou:
            best_val_miou = val_miou
            best_state = copy.deepcopy(model.state_dict())
        
        topk_heap = save_topk_checkpoint(
            model=model,
            epoch=epoch,
            train_miou=train_miou,
            val_miou=val_miou,
            history=history,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            ckpt_dir=CKPT_TOPK_DIR,
            top_k=10,
            heap_state=topk_heap,
        )

        if verbose:
            dt = time.time() - t0
            print(
                f"Epoch {epoch:02d}/{epochs} | "
                f"train_loss={train_loss:.4f} train_mIoU={train_miou:.4f} | "
                f"val_loss={val_loss:.4f} val_mIoU={val_miou:.4f} | "
                f"lr={current_lr:.2e} | time={dt:.1f}s"
            )

        if plot_live:
            clear_output(wait=True)
            _plot_history()

    model.load_state_dict(best_state)

    if not plot_live:
        _plot_history()

    return model, history, best_val_miou, topk_heap


In [ ]:
ENCODER_LR = 1e-5
HEAD_LR = 1e-3

WEIGHT_DECAY = 1e-4

EPOCHS = 8
WARM_EPOCHS = 8

encoder_params = [p for p in m_tp.encoder.parameters() if p.requires_grad]
encoder_param_ids = {id(p) for p in encoder_params}

other_params = [
    p for p in m_tp.parameters()
    if p.requires_grad and id(p) not in encoder_param_ids
]

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": other_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

criterion = torch.nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

scheduler = torch.optim.lr_scheduler.ConstantLR(optimizer, factor=1.0, total_iters=WARM_EPOCHS)

# scheduler = torch.optim.lr_scheduler.SequentialLR(
#     optimizer,
#     schedulers=[
#         torch.optim.lr_scheduler.ConstantLR(
#             optimizer,
#             factor=1.0,
#             total_iters=WARM_EPOCHS,
#         ),
#         torch.optim.lr_scheduler.CosineAnnealingLR(
#             optimizer,
#             T_max=EPOCHS - WARM_EPOCHS,
#             eta_min=1e-5,
#         ),
#     ],
#     milestones=[WARM_EPOCHS],
# )

scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16) if USE_FP16 else None

In [ ]:
m_tp, history, best_val_miou, topk_heap = learning_loop(
    model=m_tp,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    device=DEVICE,
    scaler=scaler,
    scheduler=scheduler,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
)

In [ ]:
print("Best val mIoU:", round(best_val_miou, 4))

In [ ]:
plot_compare_model(
    baseline=baseline,
    modified_model=m_tp,
    loader=val_loader,
    device=DEVICE,
    class_names=val_ds.class_names,
    per_class_iou_fn=per_class_iou
)